In [ ]:
!pip install "datasets<4.0.0" transformers accelerate librosa soundfile

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 15.5 MB/s eta 0:00:00
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [ ]:
!pip install -q evaluate jiwer librosa soundfile --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 13.7 MB/s eta 0:00:00


In [ ]:
!pip install -q evaluate jiwer

In [ ]:
import io
import torch
import librosa
import evaluate
from dataclasses import dataclass
from datasets import load_dataset, Audio
from typing import Any, Dict, List, Union
from transformers import WhisperForConditionalGeneration, WhisperProcessor, Seq2SeqTrainingArguments, Seq2SeqTrainer

In [ ]:
from google.colab import drive
# 1. MOUNT & INSTALL
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 2. LOAD DATASET
dataset = load_dataset("Anv-ke/kikuyu", streaming=True)
dataset = dataset.cast_column("audio", Audio(decode=False))

In [ ]:
# 3. LOAD MODEL & PROCESSOR
model_id = "openai/whisper-small"
processor = WhisperProcessor.from_pretrained(model_id, language="swahili", task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained(model_id)
model.config.use_cache = False
model.generation_config.forced_decoder_ids = None


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Resolving data files:   0%|          | 0/261 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/21 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

In [ ]:
# 4. Data PREPARATION FUNCTION
def prepare_dataset(batch):
    try:
        audio_bytes = batch["audio"]["bytes"]
        # Librosa is more robust for weird headers
        y, sr = librosa.load(io.BytesIO(audio_bytes), sr=16000)

        batch["input_features"] = processor.feature_extractor(y, sampling_rate=16000).input_features[0] #turns the audio into a log_mel spectogram
        batch["labels"] = processor.tokenizer(batch["transcription"]).input_ids   # converts the text transcriptions into "Token IDs" (numbers) for the model to predict.
        return batch
    except Exception as e:
        print(f" Bypassed broken sample: {e}")
        return {} # Return None for the filter to catch


dataset_processed = dataset.map(prepare_dataset)   #tokenization
# This removes the empty {} entries before they reach the model
dataset_processed = dataset_processed.filter(lambda x: len(x) > 0)

train_dataset = dataset_processed["train"].skip(50)
eval_dataset = dataset_processed["train"].take(50)


In [ ]:
# METRICS
metric = evaluate.load("wer")
def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    return {"wer": 100 * metric.compute(predictions=pred_str, references=label_str)}


In [ ]:
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # 1. Filter out None or empty features again just to be safe
        features = [f for f in features if f is not None and len(f) > 0 and "input_features" in f]

        # If the entire batch is empty because of a broken file,
        # we can't return nothing. We throw a custom warning and return a dummy sample
        # from the processor to keep the loop alive.
        if len(features) == 0:
            # This is a rare edge case where a whole batch is corrupted.
            # We return a tiny bit of silence to keep the GPU from crashing.
            import numpy as np
            silence = np.zeros(16000)
            dummy_input = self.processor.feature_extractor(silence, sampling_rate=16000).input_features[0]
            dummy_label = [self.processor.tokenizer.pad_token_id]
            features = [{"input_features": dummy_input, "labels": dummy_label}]

        # 3. Normal padding logic
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        batch["labels"] = labels
        return batch

# 8. TRAINING ARGUMENTS
training_args = Seq2SeqTrainingArguments(
    output_dir="/content/drive/MyDrive/whisper_kikuyu_final",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=5e-6,
    warmup_steps=50,
    max_steps=2000,
    gradient_checkpointing=True,
    fp16=True,
    eval_strategy="steps",
    eval_steps=200,      # Show WER every 200
    logging_steps=25,    # Show Training Loss every 25
    save_steps=200,
    save_total_limit=2,
    predict_with_generate=True,
    report_to=["tensorboard"],
)

# 9. START
trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=DataCollatorSpeechSeq2SeqWithPadding(processor=processor),
    compute_metrics=compute_metrics,
)

print(" Resuming final stretch. WER tracking enabled, bad-file protection active.")
#trainer.train()
trainer.train(resume_from_checkpoint=True)

🚀 Resuming final stretch. WER tracking enabled, bad-file protection active.


There were missing keys in the checkpoint model loaded: ['proj_out.weight'].
	logging_steps: 25 (from args) != 500 (from trainer_state.json)
'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/Anv-ke/kikuyu/resolve/c4c9e5ff2a26b3e73d060650f47e5504c14ac3ae/train/scripted/audios/train_scripted_006.parquet
Retrying in 1s [Retry 1/5].


✅ Bypassed broken sample: Error opening <_io.BytesIO object at 0x7f374da7ec00>: Format not recognised.


Step,Training Loss,Validation Loss,Wer
1200,8.388008,0.416211,53.142329
1400,8.388008,0.354929,31.977819
1600,7.682335,0.337114,31.977819
1800,7.682335,0.309406,29.852126
2000,6.892108,0.285495,28.003697


✅ Bypassed broken sample: Error opening <_io.BytesIO object at 0x7f374d6f2a70>: Format not recognised.


...

✅ Bypassed broken sample: Error opening <_io.BytesIO object at 0x7f374e8d3420>: Format not recognised.
✅ Bypassed broken sample: Error opening <_io.BytesIO object at 0x7f374dc8b290>: Format not recognised.
✅ Bypassed broken sample: Error opening <_io.BytesIO object at 0x7f375b5c2bb0>: Format not recognised.
✅ Bypassed broken sample: Error opening <_io.BytesIO object at 0x7f374e8ddda0>: Format not recognised.
✅ Bypassed broken sample: Error opening <_io.BytesIO object at 0x7f374d6d1e40>: Format not recognised.
✅ Bypassed broken sample: Error opening <_io.BytesIO object at 0x7f37b3cd5260>: Format not recognised.
✅ Bypassed broken sample: Error opening <_io.BytesIO object at 0x7f374e8d2890>: Format not recognised.
✅ Bypassed broken sample: Error opening <_io.BytesIO object at 0x7f374d6d8270>: Format not recognised.
✅ Bypassed broken sample: Error opening <_io.BytesIO object at 0x7f374d6da890>: Format not recognised.
✅ Bypassed broken sample: Error opening <_io.BytesIO object at 0x7f374199

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Bypassed broken sample: Error opening <_io.BytesIO object at 0x7f374da08180>: Format not recognised.
✅ Bypassed broken sample: Error opening <_io.BytesIO object at 0x7f374dd72890>: Format not recognised.
✅ Bypassed broken sample: Error opening <_io.BytesIO object at 0x7f37419e3970>: Format not recognised.
✅ Bypassed broken sample: Error opening <_io.BytesIO object at 0x7f37419e1710>: Format not recognised.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2000, training_loss=3.6436109619140624, metrics={'train_runtime': 6165.5655, 'train_samples_per_second': 5.19, 'train_steps_per_second': 0.324, 'total_flos': 9.23473281024e+18, 'train_loss': 3.6436109619140624, 'epoch': 1.0})

In [ ]:
# Create a final directory
final_path = "/content/drive/MyDrive/whisper_model_kikuyu_FINAL"

# Save only the model and the processor (much smaller than a checkpoint)
trainer.save_model(final_path)
processor.save_pretrained(final_path)

print(" Model saved to {final_path}. You can now delete the 'checkpoint-xxx' folders to save space!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model saved to /content/drive/MyDrive/whisper_model_kikuyu_FINAL. You can now delete the 'checkpoint-xxx' folders to save space!


In [ ]:
# 1. Load Model and Processor
model_path = "/content/drive/MyDrive/whisper_kikuyu_FINAL"
processor = WhisperProcessor.from_pretrained(model_path)
model = WhisperForConditionalGeneration.from_pretrained(model_path).to("cuda")

# 2. Load Audio manually (Standardizes the format)
audio_file = "/content/Evaline_Kahoro_316.webm"
speech, _ = librosa.load(audio_file, sr=16000)
input_features = processor(speech, sampling_rate=16000, return_tensors="pt").input_features.to("cuda")

# 3. Generate Transcription
# We use 'swahili' as a proxy OR just remove the language tag
predicted_ids = model.generate(
    input_features,
    forced_decoder_ids=processor.get_decoder_prompt_ids(language="swahili", task="transcribe")
)

# 4. Decode
transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

print("-" * 30)
print("KIKUYU TRANSCRIPTION:")
print(transcription)
print("-" * 30)

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

/tmp/ipython-input-2123965579.py:8: UserWarning: PySoundFile failed. Trying audioread instead.
  speech, _ = librosa.load(audio_file, sr=16000)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


------------------------------
KIKUYU TRANSCRIPTION:
Nĩ kũrĩ na ũthundũri mũingĩ na makĩria ya mwena wa ũgima wa mwĩrĩ, tarĩ macini cia mũndũ gũthũri mũkara arĩ na kansa ĩkamenyeka tene nĩ ĩgakorwo ya kũrihĩtwo. O na ningĩ, macini cia gũthambia mwĩrĩ, mĩrĩa cĩta gũcenda ya LAYSES rĩrĩa higo ciaku ciarĩ mo, nĩ kũruta wĩra. Ma ningĩ macini cia kũrora hihi mwana waku arĩ ndainĩ nĩ wa kahĩka nĩ wa kairĩtu. Maũndũ macio mothe nĩ makoretwo mĩ ma bata.
------------------------------


In [ ]:
# 1. Load Model and Processor
model_path = "/content/drive/MyDrive/whisper_kikuyu_FINAL"
processor = WhisperProcessor.from_pretrained(model_path)
model = WhisperForConditionalGeneration.from_pretrained(model_path).to("cuda")

# 2. Load Audio manually (Standardizes the format)
audio_file = "/content/Evaline_Kahoro_320.webm"
speech, _ = librosa.load(audio_file, sr=16000)
input_features = processor(speech, sampling_rate=16000, return_tensors="pt").input_features.to("cuda")

# 3. Generate Transcription
# We use 'swahili' as a proxy OR just remove the language tag
predicted_ids = model.generate(
    input_features,
    forced_decoder_ids=processor.get_decoder_prompt_ids(language="swahili", task="transcribe")
)

# 4. Decode
transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

print("-" * 30)
print("KIKUYU TRANSCRIPTION:")
print(transcription)
print("-" * 30)

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

/tmp/ipython-input-3405583048.py:8: UserWarning: PySoundFile failed. Trying audioread instead.
  speech, _ = librosa.load(audio_file, sr=16000)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


------------------------------
KIKUYU TRANSCRIPTION:
Nĩ ũkoraga atĩ rĩrĩa mũndũ kana mũndũ wĩna kĩrĩmĩrĩre araria, kaingĩ nĩ ũkoraga atĩ mũndũ ũcio nĩ araitwo, akĩmĩria kĩndũ ũgatuĩka nĩ aitwo, ona kana ningĩ ĩgatũma agĩe na thĩna wa mũmerũ, akora nĩ araba mũmerũ, ona kana nyingĩ akaagia Ngai, na kũgũga kora atĩ no nginya arĩ nguo, ma mandagĩtarĩ maugaga atĩ tũnginya mũndũ arĩ nguo kĩrĩmi, ona gũkorwo mathĩna ma ũtikio kĩrarĩhe.
------------------------------


In [ ]:
# 1. Load Model and Processor
model_path = "/content/drive/MyDrive/whisper_kikuyu_FINAL"
processor = WhisperProcessor.from_pretrained(model_path)
model = WhisperForConditionalGeneration.from_pretrained(model_path).to("cuda")

# 2. Load Audio manually (Standardizes the format)
audio_file = "/content/voice_message_0007a6dcc3ee47358a55910d2da5755c_21072024034041.wav"
speech, _ = librosa.load(audio_file, sr=16000)
input_features = processor(speech, sampling_rate=16000, return_tensors="pt").input_features.to("cuda")

# 3. Generate Transcription
# We use 'swahili' as a proxy OR just remove the language tag
predicted_ids = model.generate(
    input_features,
    forced_decoder_ids=processor.get_decoder_prompt_ids(language="swahili", task="transcribe")
)

# 4. Decode
transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

print("-" * 30)
print("KIKUYU TRANSCRIPTION:")
print(transcription)
print("-" * 30)

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

------------------------------
KIKUYU TRANSCRIPTION:
Mũthingi na Bolo, ibatarĩkaine makĩria mũtĩ ĩgĩa ĩkĩra kĩro, nĩguo mbegũ cĩĩbangĩ wega na magetha maingĩ.
------------------------------


In [ ]:
# 1. Install missing helper libraries first
!pip install rfc3987 jiwer gradio --upgrade


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.2/24.2 MB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.7/55.7 kB 3.2 MB/s eta 0:00:00
  Attempting uninstall: gradio-client
    Found existing installation: gradio_client 1.14.0
    Uninstalling gradio_client-1.14.0:
      Successfully uninstalled gradio_client-1.14.0
  Attempting uninstall: gradio
    Found existing installation: gradio 5.50.0
    Uninstalling gradio-5.50.0:
      Successfully uninstalled gradio-5.50.0


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


✅ Model loaded successfully!


ImportError: cannot import name 'ServerReloader' from 'gradio.utils' (/usr/local/lib/python3.12/dist-packages/gradio/utils.py)

In [ ]:
import gradio as gr
from transformers import pipeline
from jiwer import wer
import torch
import nest_asyncio
import os

# 2. Apply the "Event Loop" patch
nest_asyncio.apply()

model_path = "/content/drive/MyDrive/whisper_kikuyu_FINAL"

# Verification
if not os.path.exists(model_path):
    print(" Model path not found. Please check your Drive mount.")
else:
    try:
        asr_pipe = pipeline(
            "automatic-speech-recognition",
            model=model_path,
            chunk_length_s=30,
            device="cuda" if torch.cuda.is_available() else "cpu"
        )
        print(" Model loaded successfully!")
    except Exception as e:
        print(f"Failed to load model: {e}")

def transcribe_and_evaluate(audio, reference_text):
    if audio is None: return "No audio", "0%"
    try:
        result = asr_pipe(audio)
        prediction = result["text"]

        if reference_text and len(reference_text.strip()) > 0:
            error_rate = wer(reference_text.lower().strip(), prediction.lower().strip())
            return prediction, f"{error_rate * 100:.2f}%"
        return prediction, "N/A"
    except Exception as e:
        return f"Error: {str(e)}", "Error"

# Creating the Interface
demo = gr.Interface(
    fn=transcribe_and_evaluate,
    inputs=[
        gr.Audio(type="filepath", label="Input Audio"),
        gr.Textbox(label="Reference Text")
    ],
    outputs=[
        gr.Textbox(label="Prediction", lines=10),
        gr.Label(label="WER")
    ],
    title="OpenAi Whisper Model Small",
    flagging_mode="never"
)

# 4. Launch
demo.launch(share=True, debug=True)

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


 Model loaded successfully!
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://ee1c102f5ea1979c2a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'tr